# CET3013 Deep Learning — Task 1
## Building and Training a CNN from Scratch on MNIST

**University of Sunderland | Module: Deep Learning**

---

### Pipeline Overview
1. Install & Import Libraries
2. Load & Preprocess MNIST Dataset
3. Visualise Sample Images
4. Define CNN Architecture (Baseline)
5. Train the Baseline Model
6. Evaluate & Plot Learning Curves
7. Systematic Experiments (Architectural Variations)
8. Compare All Models
9. Final Best Model Evaluation (Confusion Matrix + Classification Report)

---
## Step 1 — Install & Import Libraries

In [ ]:
# Install any missing packages (uncomment if needed on Colab)
# !pip install torch torchvision matplotlib seaborn scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Check if GPU is available (important for speed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## Step 2 — Load & Preprocess MNIST Dataset

MNIST contains 70,000 greyscale 28x28 images of handwritten digits (0-9).
- **Training set:** 60,000 images
- **Test set:** 10,000 images

**Preprocessing applied:**
- `ToTensor()` — converts PIL image to PyTorch tensor and scales pixel values from [0,255] to [0,1]
- `Normalize((0.1307,), (0.3081,))` — normalises using the known MNIST mean and std to centre the data

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Download and load training set
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Download and load test set
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders (batches data for efficient training)
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,      # Shuffle training data each epoch
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,     # No need to shuffle test data
    num_workers=2
)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples:     {len(test_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Classes: {train_dataset.classes}')
print(f'Image shape: {train_dataset[0][0].shape}')  # Should be [1, 28, 28]

---
## Step 3 — Visualise Sample Images

Always visualise your data before training. This verifies loading is correct and builds intuition about the task.

In [ ]:
# Display a grid of sample images
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('MNIST Dataset — Sample Images (one per digit class)', fontsize=14)

# Collect one sample per class
class_samples = {i: [] for i in range(10)}
for img, label in train_dataset:
    if len(class_samples[label]) < 3:
        class_samples[label].append(img)
    if all(len(v) == 3 for v in class_samples.values()):
        break

for row in range(3):
    for col in range(10):
        ax = axes[row, col]
        img = class_samples[col][row].squeeze().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'Digit: {col}', fontsize=10)

plt.tight_layout()
plt.savefig('mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample visualisation saved as mnist_samples.png')

---
## Step 4 — Define CNN Architecture (Baseline Model)

The baseline is a simple 2-layer CNN:

```
Input [1x28x28]
  → Conv2D(1→32, 3x3) + ReLU + MaxPool(2x2)  → [32x13x13]
  → Conv2D(32→64, 3x3) + ReLU + MaxPool(2x2) → [64x5x5]
  → Flatten → [1600]
  → FC(1600→128) + ReLU
  → FC(128→10)
  → Output (10 classes)
```

**Why these choices?**
- `Conv2D`: Learns spatial features (edges, curves) from the image
- `ReLU`: Introduces non-linearity; computationally efficient (LeCun et al., 1998)
- `MaxPool`: Reduces spatial dimensions, provides translation invariance
- `FC layers`: Maps extracted features to class scores

In [ ]:
class BaselineCNN(nn.Module):
    """
    Baseline CNN: 2 convolutional layers + 2 fully connected layers.
    This serves as the starting point for architectural experiments.
    """
    def __init__(self):
        super(BaselineCNN, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

        # Convolutional Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        # Fully Connected Layers
        # After 2x MaxPool on 28x28: 28 -> 14 -> 7, so 64 * 7 * 7 = 3136
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 output classes (digits 0-9)

    def forward(self, x):
        # Block 1: Conv -> ReLU -> Pool
        x = self.pool(F.relu(self.conv1(x)))

        # Block 2: Conv -> ReLU -> Pool
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # No softmax here — CrossEntropyLoss handles it

        return x


# Instantiate and inspect the model
baseline_model = BaselineCNN().to(device)
print(baseline_model)

# Count total parameters
total_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

---
## Step 5 — Training & Evaluation Functions

We define reusable functions so we can easily train multiple model variants.

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.001, model_name='Model'):
    """
    Trains a model and records loss/accuracy per epoch.
    Returns: dict with training history.
    """
    # Loss function: CrossEntropyLoss is standard for multi-class classification
    criterion = nn.CrossEntropyLoss()

    # Optimiser: Adam — adaptive learning rate, widely used (Kingma & Ba, 2015)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss':  [], 'test_acc':  []
    }

    for epoch in range(epochs):
        # ---- Training Phase ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          # Clear gradients
            outputs = model(images)        # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()               # Backward pass
            optimizer.step()              # Update weights

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc  = 100. * correct / total

        # ---- Evaluation Phase ----
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():  # No gradient computation during evaluation
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_loss /= len(test_loader)
        test_acc   = 100. * correct / total

        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        print(f'[{model_name}] Epoch {epoch+1:2d}/{epochs} | '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | '
              f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')

    print(f'\n✅ Final Test Accuracy for {model_name}: {history["test_acc"][-1]:.2f}%\n')
    return history


def plot_learning_curves(history, model_name='Model'):
    """Plots accuracy and loss curves for a trained model."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_acc']) + 1)

    # Accuracy plot
    ax1.plot(epochs, history['train_acc'], 'b-o', label='Train Accuracy')
    ax1.plot(epochs, history['test_acc'],  'r-o', label='Test Accuracy')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss plot
    ax2.plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
    ax2.plot(epochs, history['test_loss'],  'r-o', label='Test Loss')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = f'{model_name.replace(" ", "_")}_curves.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Learning curves saved as {filename}')


print('Training and evaluation functions defined.')

---
## Step 6 — Train the Baseline Model

In [ ]:
EPOCHS = 10

print('=' * 70)
print('Training Baseline CNN (2 conv layers, 32/64 filters, ReLU, MaxPool)')
print('=' * 70)

baseline_model = BaselineCNN().to(device)
baseline_history = train_model(
    baseline_model, train_loader, test_loader,
    epochs=EPOCHS, lr=0.001, model_name='Baseline CNN'
)

plot_learning_curves(baseline_history, 'Baseline CNN')

---
## Step 7 — Systematic Experiments

We now systematically vary ONE architectural component at a time.
This is the evidence-based approach required by the assignment.

| Experiment | What changes | Why? |
|---|---|---|
| Exp 1 | More filters (64/128) | Test effect of wider network |
| Exp 2 | Deeper network (3 conv layers) | Test effect of depth |
| Exp 3 | Batch Normalisation added | Stabilises training (Ioffe & Szegedy, 2015) |
| Exp 4 | Dropout added | Reduces overfitting (Srivastava et al., 2014) |
| Exp 5 | LeakyReLU instead of ReLU | Addresses dying ReLU problem |

---
### Experiment 1 — More Filters (64/128)

In [ ]:
class WiderCNN(nn.Module):
    """Experiment 1: More filters (64 and 128) vs baseline (32 and 64)."""
    def __init__(self):
        super(WiderCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)   # 64 filters (was 32)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1) # 128 filters (was 64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(128 * 7 * 7, 256)
        self.fc2   = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print('=' * 70)
print('Experiment 1: Wider CNN (64/128 filters)')
print('=' * 70)
exp1_model   = WiderCNN().to(device)
exp1_history = train_model(exp1_model, train_loader, test_loader,
                            epochs=EPOCHS, lr=0.001, model_name='Exp1 Wider CNN')
plot_learning_curves(exp1_history, 'Exp1 Wider CNN')

---
### Experiment 2 — Deeper Network (3 Conv Layers)

In [ ]:
class DeeperCNN(nn.Module):
    """Experiment 2: 3 convolutional layers instead of 2."""
    def __init__(self):
        super(DeeperCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)  # Extra layer
        self.pool  = nn.MaxPool2d(2, 2)
        # After 3 pools on 28x28: 28->14->7->3, so 128*3*3=1152
        self.fc1   = nn.Linear(128 * 3 * 3, 256)
        self.fc2   = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28 -> 14
        x = self.pool(F.relu(self.conv2(x)))  # 14 -> 7
        x = self.pool(F.relu(self.conv3(x)))  # 7  -> 3
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print('=' * 70)
print('Experiment 2: Deeper CNN (3 conv layers)')
print('=' * 70)
exp2_model   = DeeperCNN().to(device)
exp2_history = train_model(exp2_model, train_loader, test_loader,
                            epochs=EPOCHS, lr=0.001, model_name='Exp2 Deeper CNN')
plot_learning_curves(exp2_history, 'Exp2 Deeper CNN')

---
### Experiment 3 — Batch Normalisation

Batch Normalisation normalises the input to each layer, stabilising training and often improving accuracy (Ioffe & Szegedy, 2015).

In [ ]:
class BatchNormCNN(nn.Module):
    """Experiment 3: Add Batch Normalisation after each conv layer."""
    def __init__(self):
        super(BatchNormCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)   # Batch Norm after conv1
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)   # Batch Norm after conv2
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.bn_fc = nn.BatchNorm1d(128)  # Batch Norm after fc1
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # Conv -> BN -> ReLU -> Pool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.fc2(x)
        return x

print('=' * 70)
print('Experiment 3: CNN with Batch Normalisation')
print('=' * 70)
exp3_model   = BatchNormCNN().to(device)
exp3_history = train_model(exp3_model, train_loader, test_loader,
                            epochs=EPOCHS, lr=0.001, model_name='Exp3 BatchNorm CNN')
plot_learning_curves(exp3_history, 'Exp3 BatchNorm CNN')

---
### Experiment 4 — Dropout

Dropout randomly disables neurons during training, acting as a regulariser to reduce overfitting (Srivastava et al., 2014).

In [ ]:
class DropoutCNN(nn.Module):
    """Experiment 4: Add Dropout (0.5) before fully connected layers."""
    def __init__(self):
        super(DropoutCNN, self).__init__()
        self.conv1   = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2   = nn.Conv2d(32, 64, 3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=0.5)   # 50% dropout
        self.fc1     = nn.Linear(64 * 7 * 7, 128)
        self.fc2     = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)           # Apply dropout before FC
        x = F.relu(self.fc1(x))
        x = self.dropout(x)           # Apply dropout between FC layers
        x = self.fc2(x)
        return x

print('=' * 70)
print('Experiment 4: CNN with Dropout (p=0.5)')
print('=' * 70)
exp4_model   = DropoutCNN().to(device)
exp4_history = train_model(exp4_model, train_loader, test_loader,
                            epochs=EPOCHS, lr=0.001, model_name='Exp4 Dropout CNN')
plot_learning_curves(exp4_history, 'Exp4 Dropout CNN')

---
### Experiment 5 — LeakyReLU Activation

LeakyReLU allows a small gradient when the unit is not active, addressing the 'dying ReLU' problem where neurons can become permanently inactive.

In [ ]:
class LeakyReLUCNN(nn.Module):
    """Experiment 5: Replace ReLU with LeakyReLU."""
    def __init__(self):
        super(LeakyReLUCNN, self).__init__()
        self.conv1     = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2     = nn.Conv2d(32, 64, 3, padding=1)
        self.pool      = nn.MaxPool2d(2, 2)
        self.leaky     = nn.LeakyReLU(negative_slope=0.1)  # Allows small negative gradients
        self.fc1       = nn.Linear(64 * 7 * 7, 128)
        self.fc2       = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.leaky(self.conv1(x)))
        x = self.pool(self.leaky(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.leaky(self.fc1(x))
        x = self.fc2(x)
        return x

print('=' * 70)
print('Experiment 5: CNN with LeakyReLU activation')
print('=' * 70)
exp5_model   = LeakyReLUCNN().to(device)
exp5_history = train_model(exp5_model, train_loader, test_loader,
                            epochs=EPOCHS, lr=0.001, model_name='Exp5 LeakyReLU CNN')
plot_learning_curves(exp5_history, 'Exp5 LeakyReLU CNN')

---
## Step 8 — Compare All Models

Summarise the final test accuracy of all architectures in a single bar chart.

In [ ]:
# Collect results from all experiments
results = {
    'Baseline CNN':      baseline_history['test_acc'][-1],
    'Wider CNN':         exp1_history['test_acc'][-1],
    'Deeper CNN':        exp2_history['test_acc'][-1],
    'BatchNorm CNN':     exp3_history['test_acc'][-1],
    'Dropout CNN':       exp4_history['test_acc'][-1],
    'LeakyReLU CNN':     exp5_history['test_acc'][-1],
}

# Sort by accuracy
results_sorted = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

# Plot bar chart
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if v == max(results.values()) else '#3498db'
          for v in results_sorted.values()]

bars = ax.bar(results_sorted.keys(), results_sorted.values(), color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, val in zip(bars, results_sorted.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() - 0.5,
            f'{val:.2f}%', ha='center', va='top', fontweight='bold', color='white', fontsize=11)

ax.set_ylim([min(results.values()) - 2, 100])
ax.set_title('Test Accuracy Comparison — All Architectural Variations', fontsize=14, fontweight='bold')
ax.set_xlabel('Model Architecture')
ax.set_ylabel('Test Accuracy (%)')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Results Summary ---')
for name, acc in results_sorted.items():
    marker = ' ← BEST' if acc == max(results.values()) else ''
    print(f'{name:<25} {acc:.2f}%{marker}')

---
## Step 9 — Final Best Model Evaluation

Evaluate the best-performing model in detail using:
- Confusion matrix
- Per-class precision, recall, F1-score
- Sample predictions visualisation

**Note:** Replace `best_model` below with whichever model performed best in Step 8.

In [ ]:
# ===== SELECT YOUR BEST MODEL HERE =====
# Replace this with whichever model had the highest test accuracy
best_model = exp3_model   # Example: BatchNorm was best
best_name  = 'BatchNorm CNN'
# ========================================

best_model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Classification Report ---
print(f'\nClassification Report — {best_name}')
print('=' * 60)
print(classification_report(all_labels, all_preds,
                             target_names=[str(i) for i in range(10)]))

In [ ]:
# --- Visualise Sample Predictions ---
best_model.eval()

images_batch, labels_batch = next(iter(test_loader))
images_batch = images_batch.to(device)

with torch.no_grad():
    outputs = best_model(images_batch)
    _, predicted = outputs.max(1)

fig, axes = plt.subplots(3, 8, figsize=(16, 7))
fig.suptitle(f'Sample Predictions — {best_name}\n(Green = Correct, Red = Wrong)', fontsize=13)

for idx, ax in enumerate(axes.flat):
    img   = images_batch[idx].cpu().squeeze().numpy()
    true  = labels_batch[idx].item()
    pred  = predicted[idx].cpu().item()
    color = 'green' if pred == true else 'red'

    ax.imshow(img, cmap='gray')
    ax.set_title(f'T:{true} P:{pred}', color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample predictions saved.')

---
## Summary

| Step | What was done |
|---|---|
| Data loading | MNIST loaded via torchvision, normalised |
| Baseline | 2 conv layers, 32/64 filters, ReLU, MaxPool |
| Exp 1 | Wider (64/128 filters) |
| Exp 2 | Deeper (3 conv layers) |
| Exp 3 | Batch Normalisation added |
| Exp 4 | Dropout (p=0.5) added |
| Exp 5 | LeakyReLU activation |
| Evaluation | Confusion matrix, classification report, visual predictions |

---
### References (Harvard Style)
- Ioffe, S. and Szegedy, C. (2015) 'Batch Normalization: Accelerating deep network training by reducing internal covariate shift', *Proceedings of the 32nd ICML*, pp. 448–456.
- Kingma, D.P. and Ba, J. (2015) 'Adam: A method for stochastic optimization', *ICLR 2015*. Available at: https://arxiv.org/abs/1412.6980
- LeCun, Y. et al. (1998) 'Gradient-based learning applied to document recognition', *Proceedings of the IEEE*, 86(11), pp. 2278–2324.
- Srivastava, N. et al. (2014) 'Dropout: A simple way to prevent neural networks from overfitting', *Journal of Machine Learning Research*, 15(1), pp. 1929–1958.